It is for verifying the genrated internal ONNX graphs generated during the flow before the HW Conversion. The main base for the verification process is the FINN function finn.core.onnx_exec.execute_onnx

In [5]:
import glob
import torch
import numpy as np
import finn.core.onnx_exec as oxe
from PIL import Image

from qonnx.core.modelwrapper import ModelWrapper

build_dir = './finn_build'
onnx_dir = './finn_build/'

## Validating the output
In this cell you can check the output values from each generated ONNX graph and compare it with the original model.

In [6]:
test_img_paths = glob.glob("./data/-monterey_trail009_jpg.rf.a1bba9241fad77edd0dbbf396178acb1.jpg")

for number, test_img_path in enumerate(test_img_paths):
    # Open image using Pillow
    img_org = Image.open(test_img_path).convert('RGB')  # Ensure RGB format

    # Convert PIL Image to NumPy array
    img = np.array(img_org)  # shape: (H, W, 3), RGB format

    # Convert to float32 for processing
    img = img.astype(np.float32)
    img_ip = img.astype(np.uint8)

    # Prepare input for ONNX
    driver_in = np.expand_dims(img_ip, 0)  # Add batch dimension
    driver_in = driver_in.astype(np.float32)
    driver_in = driver_in.transpose(0, 3, 1, 2)  # Convert to NCHW format

    input_dict = {"global_in": driver_in}

    # Simulate inference
    model_for_sim = ModelWrapper(onnx_dir + "yolov1-tiny_streamlined.onnx")
    output_dict = oxe.execute_onnx(model_for_sim, input_dict, return_full_exec_context=False)

    # Post-processing
    output_pysim = output_dict[list(output_dict.keys())[0]]
    output_pysim = output_pysim.transpose(0, 2, 3, 1)
    output_pysim_arr = np.array(output_pysim)

    conf = output_pysim_arr[0, :, :, 10]/2 + 0.5
    conf *= (output_pysim_arr[0, :, :, 11]/2 + 0.5)

    print(conf.max())

0.15354331
